In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = pd.read_csv('../data/raw/credit_card_fraud_2026.csv')
df.shape

(20000, 26)

In [2]:
#Drop the ID column
#since transaction_id is a pure identifier with no predictive value which was confirmed by its 0 correlation with is_fraud
df = df.drop(columns=['transaction_id'])

In [5]:
#Check check the dataset's boolean columns actual dtype
#confirm all the Yes/No-style flags are properly boolean/numeric before encoding
bool_cols = ['is_foreign_transaction', 'ip_country_mismatch', 'billing_shipping_mismatch',
             'is_ai_generated_scam_attempt', 'used_vpn', 'is_new_merchant', 'is_fraud']

for col in bool_cols:
    print(col, '->', df[col].dtype, df[col].unique()[:5])

is_foreign_transaction -> int64 [0 1]
ip_country_mismatch -> int64 [0 1]
billing_shipping_mismatch -> int64 [0 1]
is_ai_generated_scam_attempt -> int64 [0 1]
used_vpn -> int64 [0 1]
is_new_merchant -> int64 [0 1]
is_fraud -> int64 [0 1]


In [6]:
#Convert booleans to clean 0/1 integers
for col in bool_cols:
    df[col] = df[col].astype(int)

In [7]:
#Encode the categorical columns
#the dataset contains 5 categorical columns merchant_category, card_type, auth_method, channel, device_type
#Since none of these are ordinal (there's no natural ranking), one hot encoding is more appropriate than label encoding here
cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(df_encoded.shape)
df_encoded.head()

(20000, 50)


,amount_usd,is_foreign_transaction,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,is_new_merchant,used_vpn,...,channel_In-App,channel_Online,channel_POS,device_type_Android Phone,device_type_Mac,device_type_POS Terminal,device_type_Smart Watch,device_type_Tablet,device_type_Windows PC,device_type_iPhone
0,42.86,0,13.54,2,22.35,39,25,1080.71,0,0,...,False,True,False,True,False,False,False,False,False,False
1,4.75,0,0.71,2,35.28,47,33,4463.72,1,0,...,False,True,False,True,False,False,False,False,False,False
2,77.18,0,0.35,5,9.82,51,59,366.25,0,1,...,False,True,False,False,True,False,False,False,False,False
3,1.69,0,3.42,6,19.72,48,28,1345.29,1,0,...,False,False,True,True,False,False,False,False,False,False
4,261.68,0,2.43,2,17.68,43,36,4192.59,0,0,...,True,False,False,False,False,False,False,False,False,True


In [8]:
#Separate features and target
X = df_encoded.drop(columns=['is_fraud'])
y = df_encoded['is_fraud']
print(X.shape, y.shape)

(20000, 49) (20000,)


In [9]:
#Scale the numeric features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_scaled.describe().round(3)

,amount_usd,is_foreign_transaction,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,is_new_merchant,used_vpn,...,channel_In-App,channel_Online,channel_POS,device_type_Android Phone,device_type_Mac,device_type_POS Terminal,device_type_Smart Watch,device_type_Tablet,device_type_Windows PC,device_type_iPhone
count,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,...,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000
mean,-0.000,-0.000,0.000,-0.000,-0.000,-0.000,-0.000,0.000,0.000,-0.000,...,0.000,-0.000,0.000,-0.000,0.000,-0.000,-0.000,-0.000,0.000,0.000
std,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,...,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
min,-0.511,-0.257,-1.012,-1.794,-1.001,-3.684,-1.712,-0.750,-0.547,-0.311,...,-0.438,-0.719,-0.592,-0.616,-0.270,-0.433,-0.230,-0.248,-0.389,-0.514
25%,-0.413,-0.257,-0.720,-0.670,-0.714,-0.729,-0.847,-0.528,-0.547,-0.311,...,-0.438,-0.719,-0.592,-0.616,-0.270,-0.433,-0.230,-0.248,-0.389,-0.514
50%,-0.292,-0.257,-0.310,-0.108,-0.300,0.009,0.018,-0.301,-0.547,-0.311,...,-0.438,-0.719,-0.592,-0.616,-0.270,-0.433,-0.230,-0.248,-0.389,-0.514
75%,-0.003,-0.257,0.394,0.454,0.392,0.600,0.883,0.144,-0.547,-0.311,...,-0.438,1.392,1.689,1.623,-0.270,-0.433,-0.230,-0.248,-0.389,-0.514
max,26.231,3.890,8.836,4.950,8.772,3.998,1.694,28.458,1.829,3.220,...,2.281,1.392,1.689,1.623,3.704,2.308,4.341,4.028,2.574,1.946
